In [0]:
import logging

# 1. Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("ChicagoTaxiPipeline")

In [0]:
import pyspark.sql.functions as F
from datetime import datetime


def validate_not_empty(df, table_name):
    if len(df.take(1)) == 0:
        raise ValueError(f"{table_name} is empty")
    logger.info(f"{table_name} passed")


def validate_duplicates(df, column_name):
    duplicates = (
        df.groupBy(column_name)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    if duplicates > 0:
        raise ValueError(f"validate_duplicates - FAILED (Duplicates found in {column_name})")
    logger.info("validate_duplicates - PASSED (No duplicates found)")


def validate_schema(df, required_columns):
    """Validate that required columns exist in dataframe"""
    missing_cols = set(required_columns) - set(df.columns)
    if missing_cols:
        raise ValueError(f"validate_schema - FAILED (Missing required columns: {missing_cols})")
    logger.info(f"validate_schema - PASSED (All required columns present: {required_columns})")


def validate_no_nulls(df, column_name):
    """Validate that a column has no null values"""
    null_count = df.filter(F.col(column_name).isNull()).count()
    if null_count > 0:
        raise ValueError(f"validate_no_nulls - FAILED (Found {null_count} null values in {column_name})")
    logger.info(f"validate_no_nulls - PASSED (No nulls in `{column_name}` column)")

def validate_temperature_range(df, column_name, min_value= -40, max_value=50):
    """Validate that a column temperature has valid range"""
    extreme_temp_values = df.filter((F.col(column_name) < min_value) & (F.col(column_name) > max_value)).count()
    
    if extreme_temp_values > 0:
        raise ValueError(f"validate_temperature_range - FAILED (Found {extreme_temp_values} extreme temperature values in {column_name})")
    logger.info(f"validate_temperature_range - PASSED (Temperature Values are in range -40C and +50C")


def validate_data_type(df, column_name, expected_type):
    """Validate column data type (e.g., 'date', 'string', 'int')"""
    actual_type = dict(df.dtypes)[column_name]
    if actual_type != expected_type:
        raise ValueError(
            f"validate_data_type - FAILED (Column {column_name} has type {actual_type}, expected {expected_type})"
        )
    logger.info(f"validate_data_type - PASSED ({column_name} has correct type: {expected_type})")


def validate_row_count(df, table_name, min_rows):
    """Validate minimum row count threshold"""
    count = df.count()
    if count < min_rows:
        raise ValueError(
            f"validate_row_count - FAILED ({table_name} has {count} rows, expected at least {min_rows})"
        )
    logger.info(f"validate_row_count - PASSED ({table_name} has {count} rows (minimum: {min_rows})")


def validate_date_range(df, date_column, min_date=1900, max_date=2100):
    """Validate dates are within reasonable bounds"""
    
    invalid_dates = df.filter((F.col(date_column) < min_date) | (F.col(date_column) > max_date)).count()
    
    if invalid_dates > 0:
        raise ValueError(
            f"validate_date_range - FAILED (Found {invalid_dates} dates outside range {min_date}-{max_date})"
        )
    logger.info(f"validate_date_range - PASSED (All dates in column {date_column} are within {min_date}-{max_date})")


def validate_chronology(df):
    """Ensures trip ended after it started"""
    errors = df.filter(F.col("trip_start_timestamp") > F.col("trip_end_timestamp")).count()
    if errors > 0:
        raise ValueError(f"validate_chronology - FAILED ({errors} rows show trip ending before it started.)")
    logger.info("validate_chronology - PASSED")


def validate_positive_values(df, columns):
    """Ensures metrics like miles and seconds are not negative"""
    for col in columns:
        neg_count = df.filter(F.col(col) < 0).count()
        if neg_count > 0:
            raise ValueError(f"validate_positive_values - FAILED (Column '{col}' has {neg_count} negative values.)")
    logger.info(f"validate_positive_values - PASSED (Positive value check passed for: {columns})")